<a href="https://colab.research.google.com/github/tassiana-bastos/my_portfolio/blob/%F0%9F%9A%80-Big-Data-Apache-Spark/Clients_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null


In [ ]:
!wget -q https://archive.apache.org/dist/spark/spark-3.4.1/spark-3.4.1-bin-hadoop3.tgz
!ls -lh spark-3.4.1-bin-hadoop3.tgz


-rw-r--r-- 1 root root 371M Jun 19  2023 spark-3.4.1-bin-hadoop3.tgz


In [ ]:
!tar -xzf spark-3.4.1-bin-hadoop3.tgz


In [ ]:
!pip install -q findspark


In [ ]:
import os
import findspark

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.4.1-bin-hadoop3"
findspark.init()

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Clientes").getOrCreate()


In [ ]:
from google.colab import files
uploaded = files.upload()


Saving clientes.csv to clientes.csv


In [ ]:
df = spark.read.csv("clientes.csv", header=True, inferSchema=True)
df.show(5)


+-----+-------------+------------+--------------+--------------+---------+-------+
|idade| escolaridade|estado_civil|gastos_mensais|        cidade|   genero|  renda|
+-----+-------------+------------+--------------+--------------+---------+-------+
|   56|Pós-graduação|      Casado|       3999.39|      Salvador| Feminino|6770.75|
|   69|        Médio|    Solteiro|       3518.72|Rio de Janeiro| Feminino|4595.16|
|   46|     Superior|      Casado|       2516.15|Belo Horizonte| Feminino|7223.94|
|   32|  Fundamental|      Casado|       2217.18|Rio de Janeiro|Masculino|5048.65|
|   60|     Superior|  Divorciado|       3793.75|  Porto Alegre|Masculino|6797.94|
+-----+-------------+------------+--------------+--------------+---------+-------+
only showing top 5 rows



In [ ]:
df.printSchema()


root
 |-- idade: integer (nullable = true)
 |-- escolaridade: string (nullable = true)
 |-- estado_civil: string (nullable = true)
 |-- gastos_mensais: double (nullable = true)
 |-- cidade: string (nullable = true)
 |-- genero: string (nullable = true)
 |-- renda: double (nullable = true)



In [ ]:
from pyspark.sql.functions import col, sum

df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()


+-----+------------+------------+--------------+------+------+-----+
|idade|escolaridade|estado_civil|gastos_mensais|cidade|genero|renda|
+-----+------------+------------+--------------+------+------+-----+
|    0|           0|           0|             0|     0|     0|    0|
+-----+------------+------------+--------------+------+------+-----+



In [ ]:
df.describe().show()


+-------+------------------+------------+------------+------------------+--------------+---------+------------------+
|summary|             idade|escolaridade|estado_civil|    gastos_mensais|        cidade|   genero|             renda|
+-------+------------------+------------+------------+------------------+--------------+---------+------------------+
|  count|              5000|        5000|        5000|              5000|          5000|     5000|              5000|
|   mean|           43.5846|        null|        null|2545.8105279999936|          null|     null| 5337.119908000002|
| stddev|14.919093763085689|        null|        null| 629.4652003123934|          null|     null|1666.8459299094686|
|    min|                18| Fundamental|      Casado|            771.93|Belo Horizonte| Feminino|            1000.0|
|    max|                69|    Superior|    Solteiro|           5021.55|     São Paulo|Masculino|           11748.2|
+-------+------------------+------------+------------+--

In [ ]:
df.groupBy("escolaridade").count().orderBy("count", ascending=False).show()


+-------------+-----+
| escolaridade|count|
+-------------+-----+
|        Médio| 2049|
|     Superior| 1438|
|  Fundamental| 1030|
|Pós-graduação|  483|
+-------------+-----+



In [ ]:
df.groupBy("estado_civil").count().orderBy("count", ascending=False).show()


+------------+-----+
|estado_civil|count|
+------------+-----+
|      Casado| 1691|
|    Solteiro| 1660|
|  Divorciado| 1649|
+------------+-----+



In [ ]:
df.groupBy("cidade").count().orderBy("count", ascending=False).show()


+--------------+-----+
|        cidade|count|
+--------------+-----+
|Rio de Janeiro| 1034|
|     São Paulo| 1032|
|      Salvador|  996|
|  Porto Alegre|  972|
|Belo Horizonte|  966|
+--------------+-----+



In [ ]:
print("A correlação idade vs renda é de:", df.corr("idade", "renda"))
print("A correlação idade vs gastos_mensais é de:", df.corr("idade", "gastos_mensais"))
print("A correlação gastos_mensais vs renda é de:", df.corr("gastos_mensais", "renda"))


A correlação idade vs renda é de: 0.45189836194769345
A correlação idade vs gastos_mensais é de: 0.7098362861197246
A correlação gastos_mensais vs renda é de: 0.6324383368363652


### Initial observations based on exploratory analysis:

1. **Education**
  Based on the above output, we can conclude that customers with a high school education are the most present, and those with a postgraduate degree are the least present, which can directly affect the income vs. educational level relationship.

2. **Mean age and standard deviation**  
  The average age of customers is approximately **43.6 years**, with a standard deviation (stddev) of around **14.9 years**, indicating a good age range due to the high stddev.

3. **Monthly expenses vs. Income**  
  Customers spend an average of **R$ 2,545.81** per month, while the average income is **R$ 5,337.12**. This ratio (~47%) may indicate **a moderate consumption profile**, which may even be related to the average education level of these customers **high school**, indicating that they have a basic understanding of financial education.


In [ ]:
from pyspark.sql.functions import when, col, concat_ws, round


In [ ]:
df = df.withColumn("faixa_etaria", when(col("idade") < 25, "jovem")
                                     .when((col("idade") >= 25) & (col("idade") < 45), "adulto")
                                     .when((col("idade") >= 45) & (col("idade") < 60), "meia-idade")
                                     .otherwise("idoso"))


In [ ]:
df = df.withColumn("cidade_grande", when(col("cidade").isin("São Paulo", "Rio de Janeiro"), 1).otherwise(0))


In [ ]:
df = df.withColumn("gastos_por_idade", round(col("gastos_mensais") / col("idade"), 2))


In [ ]:
df = df.withColumn("escolaridade_cidade", concat_ws("_", col("escolaridade"), col("cidade")))


### Four New Variables - Feature Engineering ###

1. **faixa_etaria**  
   We segmented and grouped ages into four groups:
   - "jovem" (under 25 years old)
   - "adulto" (from 25 to 44 years old)
   - "meia-idade" (45 to 59 years old)
   - "idoso" (over 60 years old)  
    This segmentation helps capture consumption and income patterns by life stage and verify whether there is a correlation between these attributes.

2. **cidade_grande**  
   Created as a binary variable to simplify and aid analysis, given that large cities such as São Paulo and Rio de Janeiro represent a different economic context from the other cities mentioned:
   - 1 for residents of **São Paulo** or **Rio de Janeiro**
   - 0 for other cities
   These capitals can influence average income and customer consumption.

3. **gastos_por_idade**  
   Calculated as `gastos_mensais / idade`.  
   This ratio standardizes consumption levels by age group, revealing more or less intense spending profiles.

4. **escolaridade_cidade**  
   Combination of education level and city (e.g., “Superior_SP,” “Medio_RJ”).  
   This allows us to identify interactions between the customer's education level and where they live, so we can determine whether this relationship influences their monthly income.



In [ ]:
from pyspark.sql.functions import col

def remove_outliers(df, column):
    quantiles = df.approxQuantile(column, [0.25, 0.75], 0.05)
    Q1, Q3 = quantiles
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return df.filter((col(column) >= lower_bound) & (col(column) <= upper_bound))

df = remove_outliers(df, "renda")
df = remove_outliers(df, "gastos_mensais")


In [ ]:
df.groupBy("cidade").count().orderBy("count").show()
df.groupBy("escolaridade").count().orderBy("count").show()
df.groupBy("estado_civil").count().orderBy("count").show()


+--------------+-----+
|        cidade|count|
+--------------+-----+
|Belo Horizonte|  947|
|  Porto Alegre|  959|
|      Salvador|  984|
|     São Paulo| 1003|
|Rio de Janeiro| 1009|
+--------------+-----+

+-------------+-----+
| escolaridade|count|
+-------------+-----+
|Pós-graduação|  410|
|  Fundamental| 1020|
|     Superior| 1428|
|        Médio| 2044|
+-------------+-----+

+------------+-----+
|estado_civil|count|
+------------+-----+
|  Divorciado| 1618|
|    Solteiro| 1623|
|      Casado| 1661|
+------------+-----+



In [ ]:
from pyspark.sql.functions import when


In [ ]:

df = df.withColumn(
    "cidade_reclassificada",
    when(col("cidade") == "Rio de Janeiro", "Rio de Janeiro")
    .when(col("cidade") == "São Paulo", "São Paulo")
    .otherwise("Outros")
)


### Reclassification of Cities

The variable `cidade` was reclassified based on economic relevance and representativeness in the dataset:

- The capitals **Rio de Janeiro** and **São Paulo** were kept as separate categories due to their greater significance in the data set.
- The other cities were grouped into a single category called **"Outros"**.

This restructuring simplifies data analysis and highlights urban centers with greater predictive potential in relation to income.


In [ ]:
df.groupBy("genero").count().orderBy("count", ascending=False).show()
df.groupBy("cidade_reclassificada").count().orderBy("count", ascending=False).show()
df.groupBy("estado_civil").count().orderBy("count", ascending=False).show()


+---------+-----+
|   genero|count|
+---------+-----+
| Feminino| 2518|
|Masculino| 2384|
+---------+-----+

+---------------------+-----+
|cidade_reclassificada|count|
+---------------------+-----+
|               Outros| 2890|
|       Rio de Janeiro| 1009|
|            São Paulo| 1003|
+---------------------+-----+

+------------+-----+
|estado_civil|count|
+------------+-----+
|      Casado| 1661|
|    Solteiro| 1623|
|  Divorciado| 1618|
+------------+-----+



### Evaluation of potential biases ###
After grouping the categories genero, cidade_reclassificada (addressed in the reclassification step above), and estado_civil, we can see that there is no significant difference between the variables that could negatively affect the model.

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, MinMaxScaler
from pyspark.ml import Pipeline

In [ ]:
indexers = [
    StringIndexer(inputCol=coluna, outputCol=coluna + "_index", handleInvalid="keep")
    for coluna in ["genero", "estado_civil", "escolaridade", "faixa_etaria", "cidade_reclassificada"]
]
encoders = [
    OneHotEncoder(inputCol=coluna + "_index", outputCol=coluna + "_encoded")
    for coluna in ["genero", "estado_civil", "escolaridade", "faixa_etaria", "cidade_reclassificada"]
]


In [ ]:
assembler_numeric = VectorAssembler(
    inputCols=["idade", "gastos_mensais", "gastos_por_idade"],
    outputCol="numeric_features"
)

scaler = MinMaxScaler(
    inputCol="numeric_features",
    outputCol="scaled_numeric_features"
)

In [ ]:
final_assembler = VectorAssembler(
    inputCols=[
        "scaled_numeric_features",
        "genero_encoded",
        "estado_civil_encoded",
        "escolaridade_encoded",
        "faixa_etaria_encoded",
        "cidade_reclassificada_encoded"
    ],
    outputCol="features"
)


In [ ]:
pipeline = Pipeline(stages=indexers + encoders + [assembler_numeric, scaler, final_assembler])

In [ ]:
df_preprocessado = pipeline.fit(df).transform(df)

### Enhanced Preprocessing ###

In this part, we are transforming categorical variables from text format (string) to numeric format using StringIndexer and then encoding them using the OneHotEncoder function, which can be read and interpreted by our regression model.
In addition, we are normalizing the numerical attributes (idade, gastos_mensais e gastos_por_idade) using the MinMaxScaler function.
After that, a vector is created with all the attributes encoded above with the help of the imported Pyspark functions.
Finally, we combine all the preprocessing steps (normalization of numerical data and coding of categorical attributes) in a single pipeline, which will be transformed and tested using the functions .fit e .transform.

In [ ]:
from pyspark.ml.regression import DecisionTreeRegressor
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import RegressionEvaluator

In [ ]:
regressor = DecisionTreeRegressor(featuresCol="features", labelCol="renda")


In [ ]:
paramGrid = ParamGridBuilder() \
    .addGrid(regressor.maxDepth, [3, 5, 10]) \
    .addGrid(regressor.minInstancesPerNode, [1, 5]) \
    .build()


In [ ]:
evaluator = RegressionEvaluator(
    labelCol="renda",
    predictionCol="prediction",
    metricName="rmse"
)

In [ ]:
crossval = CrossValidator(
    estimator=regressor,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3,
    seed=42
)

In [ ]:
modelo_cv = crossval.fit(df_preprocessado)


In [ ]:
melhor_modelo = modelo_cv.bestModel


### Hyperparameters and Cross-Validation ###

We chose the Decision Tree Regressor model because of its ability to capture nonlinear relationships in the data.
We performed hyperparameter tuning using 'ParamGridBuilder', testing different tree depths 'maxDepth' with values of 3, 5, and 10 and minimum number of instances per node 'minInstancesPerNode', i.e., 1, 5.
'Cross-validation' was performed with 3 folds, using RMSE as the main metric.
In the end, the model with the best performance was automatically selected for use in the evaluation stage.


In [ ]:
predicoes = modelo_cv.transform(df_preprocessado)


In [ ]:
avaliador_rmse = RegressionEvaluator(
    labelCol="renda",
    predictionCol="prediction",
    metricName="rmse"
)
rmse = avaliador_rmse.evaluate(predicoes)
print(f"RMSE: {rmse:.2f}")



RMSE: 1009.78


In [ ]:
avaliador_r2 = RegressionEvaluator(
    labelCol="renda",
    predictionCol="prediction",
    metricName="r2"
)
r2 = avaliador_r2.evaluate(predicoes)


In [ ]:
print(f"R²: {r2:.2f}")


R²: 0.60


### Model Evaluation and Justification ###
In addition to the above, the decision tree was preferred over linear regression due to its flexibility with categorical variables, in this case:  "genero", "estado_civil", "escolaridade", "faixa_etaria" e "cidade_reclassificada", thus not having the need to assume linear relationships.
After the preprocessing, hyperparameter tuning, cross-validation, and testing phases using the RMSE and R² metrics, we arrived at the following result:


- **RMSE:** 1009,78  
  The error presented here is approximately R$ 1,010 in the income forecast.

- **R²:** 0.60  
  The model explains 60% of the variation in the variable `renda`, which indicates reasonable performance, especially considering the nature of the data and predictor variables available.

These results indicate that the model learned rather than memorized, i.e., through 3-fold cross-validation and hyperparameter tuning techniques, the model achieved a percentage of 60%, which indicates that it shows reasonable signs of generalization and can be trusted for use.

In [ ]:
modelo_final = modelo_cv.bestModel


In [ ]:
pipeline_completo = Pipeline(stages=indexers + encoders + [assembler_numeric, scaler, final_assembler, modelo_final])


In [ ]:
pipeline_treinado = pipeline_completo.fit(df)


In [ ]:
pipeline_treinado.write().overwrite().save("pipeline_final_modelo")


In [ ]:
df_kmeans = df.drop("renda")


### Attribute removal 'renda' ###
Using the drop function, we are removing the ‘renda’ variable from the original df and storing the rest of the data in a copy named df_kmeans.

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, MinMaxScaler
from pyspark.ml import Pipeline

In [ ]:
indexers_kmeans = [
    StringIndexer(inputCol=coluna, outputCol=coluna + "_index", handleInvalid="keep")
    for coluna in ["genero", "estado_civil", "escolaridade", "faixa_etaria", "cidade_reclassificada"]
]

encoders_kmeans = [
    OneHotEncoder(inputCol=coluna + "_index", outputCol=coluna + "_encoded")
    for coluna in ["genero", "estado_civil", "escolaridade", "faixa_etaria", "cidade_reclassificada"]
]

In [ ]:
assembler_numeric_kmeans = VectorAssembler(
    inputCols=["idade", "gastos_mensais", "gastos_por_idade"],
    outputCol="numeric_features"
)

In [ ]:
scaler_kmeans = MinMaxScaler(
    inputCol="numeric_features",
    outputCol="scaled_numeric_features"
)

In [ ]:
final_assembler_kmeans = VectorAssembler(
    inputCols=[
        "scaled_numeric_features",
        "genero_encoded",
        "estado_civil_encoded",
        "escolaridade_encoded",
        "faixa_etaria_encoded",
        "cidade_reclassificada_encoded"
    ],
    outputCol="features"
)

In [ ]:
pipeline_kmeans = Pipeline(stages=indexers_kmeans + encoders_kmeans + [assembler_numeric_kmeans, scaler_kmeans, final_assembler_kmeans])


In [ ]:
df_kmeans_preparado = pipeline_kmeans.fit(df_kmeans).transform(df_kmeans)

### Preprocessing: indexing, encoding, and normalization ###
In this phase of preparation for clustering, we are working on the new basis, that is, the df without the 'renda' variable.
First, we indexed and coded the categorical variables. Then we transformed the numerical attributes into vectors, and then normalized them.
At the end of this part, we call both the normalized numerical attributes and the coded variables and create a new column called ‘features’.
Last but not least, we created a pipeline with all variables related to preprocessing and applied it (without the `renda` variable).


In [ ]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

In [ ]:
evaluator = ClusteringEvaluator(featuresCol="features", metricName="silhouette", distanceMeasure="squaredEuclidean")

In [ ]:
resultados = []

In [ ]:
for k in [3, 4, 5]:
    kmeans = KMeans(featuresCol="features", k=k, seed=42)
    modelo = kmeans.fit(df_kmeans_preparado)
    predicoes = modelo.transform(df_kmeans_preparado)
    silhouette = evaluator.evaluate(predicoes)
    resultados.append((k, silhouette))
    print(f"k = {k} --> Silhouette Score = {silhouette:.4f}")

k = 3 --> Silhouette Score = 0.1922
k = 4 --> Silhouette Score = 0.2235
k = 5 --> Silhouette Score = 0.2186


# Training with kmeans ###

At this stage, the evaluator is called using the ‘ClusteringEvaluator’ function, taking into account the following parameters: features (variables already processed in the pre-processing stage), the standard ‘silhouette’ metric of the KMeans method, and the distance calculated in Euclidean squared distance, also considered standard in the method in question.
Values between 3 and 5 are used to test the clusters, measuring how cohesive and how well separated they are from each other.


In [ ]:
melhor_k = 4
kmeans_final = KMeans(featuresCol="features", k=melhor_k, seed=42)
modelo_final_kmeans = kmeans_final.fit(df_kmeans_preparado)


In [ ]:
df_clusterizado = modelo_final_kmeans.transform(df_kmeans_preparado)


### Silhouette Score metric for choosing the best K ###

Given the above result, the best K is 4, with an approximate value of 0.22. Due to these transformations, a new attribute is created and stored in the df_clusterizado variable. This new attribute is called ‘prediction’ and indicates in which group each record, i.e., each customer, is stored. The maximum number of groups is calculated by K-1, i.e., 4-1, and starts at 0.



In [ ]:
from pyspark.sql.functions import avg, count, round

perfil_clusters = df_clusterizado.groupBy("prediction").agg(
    count("*").alias("num_clientes"),
    round(avg("idade"), 1).alias("idade_media"),
    round(avg("gastos_mensais"), 2).alias("gastos_mensais_medio"),
    round(avg("gastos_por_idade"), 2).alias("gastos_por_idade_medio")
).orderBy("prediction")

perfil_clusters.show()


+----------+------------+-----------+--------------------+----------------------+
|prediction|num_clientes|idade_media|gastos_mensais_medio|gastos_por_idade_medio|
+----------+------------+-----------+--------------------+----------------------+
|         0|        1310|       51.9|             2793.23|                 58.04|
|         1|        1311|       31.1|             2179.52|                  73.3|
|         2|        1207|       56.7|             2912.78|                 51.78|
|         3|        1074|       33.0|             2188.37|                 67.74|
+----------+------------+-----------+--------------------+----------------------+



### Description of 3 clusters ###

After applying the KMeans algorithm with ‘k = 4’, we obtained the following customer profiles based on demographic and consumption attributes (except income).

**Cluster 0**
 **Older adults:** with consumption **moderate and stable**. Represents mature clients, potential targets for fixed-value products, insurance, or pension plans.

**Cluster 1**  
  **Young people with high consumption relative to their age**. Ideal for entry-level products, mobility, digital services, and loyalty programs.

**Cluster 2**  
 **Older audience**, but with **highest total expenditure**. Indicates financial stability — opportunity for premium products or travel.

In the context of this work, this analysis helps the marketing team define targeted actions, offering products and communication strategies that match the behavior of each group.




In [ ]:
df_clusterizado.write.mode("overwrite").parquet("df_clusterizado.parquet")
